<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/01_master_feature_creator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LaDe Feature Pipeline — Full Run (Stage 0 → Stage 2)

**Cities:** Shanghai · Hangzhou · Chongqing  
**Target:** `eta_mins = (sign_time − receipt_time) / 60`  
**Author:** Soumya · Thesis: Causal-informed RL for ETA Prediction

**Dataset :** LaDe https://huggingface.co/datasets/Cainiao-AI/LaDe

---

## Pipeline Architecture

```
Stage 0 ── Data Preparation (this section)
    load d5c_neat.csv (combined 5-city raw data)
    ├─ map Chinese city names → English
    ├─ parse receipt_time / sign_time (prepend "2021-")
    ├─ compute eta_mins ; drop negatives
    ├─ split → shanghai_data.csv, hangzhou_data.csv, chongqing_data.csv
    ├─ identify dropped courier IDs (negative ETA couriers)
    ├─ filter GPS parquet (remove dropped couriers)
    └─ split GPS → {city}_delivery_data.parquet

Stage 1 ── Feature Engineering  (Section 2 onwards)
    per-city: GPS trajectory, batch, workload, spatial, weather, typecode

Stage 2 ── Batch Aggregation (Section 6 onwards)
    aggregate order-level features → batch-level
    ├─ identify constant batch features (e.g., city, courier_eta_ewm, WSI)
    ├─ compute batch size and diversity (e.g., batch_grid_cells_unique, batch_aoi_entropy)
    ├─ average varying order-level features (e.g., speed_mean_15m, pickup_destination_distance)
    ├─ aggregate spatial congestion (mean and std)
    ├─ aggregate target eta_mins (mean, max, std)
    └─ save per-city batch_scm_*.parquet
```

## Stage 0 · Data Preparation

Runs once on the raw combined delivery CSV. Outputs are the per-city CSV and GPS parquet files consumed by the feature pipeline in Stage 1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_RAW  = '/content/drive/MyDrive/ml/PROCESSED/matched'
CITY_DIR  = BASE_RAW + '/city_divided'

import os
os.makedirs(CITY_DIR, exist_ok=True)

import pandas as pd
import numpy as np


Mounted at /content/drive


### 0.1 Load Combined Dataset

We have 2 sets of data sources:
- delivery level data
- 20s sampled gps data dump

Here we read the cleaned five-city file (`d5c_neat.csv`). which is cleaned and sourced from
(`delivery_five_cities.csv`) directly from the LaDe dataset .

In [ ]:
# wdata = pd.read_csv(BASE_RAW + "/delivery_five_cities.csv")  # original raw
wdata = pd.read_csv(BASE_RAW + "/d5c_neat.csv")  # cleaned for the 3 cities used

print(f"Shape: {wdata.shape}")
print(f"Columns original present : {list(wdata.columns)}")
wdata.head()

Shape: (102183, 15)
Columns original present : ['order_id', 'from_dipan_id', 'from_city_name', 'delivery_user_id', 'poi_lng', 'poi_lat', 'aoi_id', 'typecode', 'receipt_time', 'receipt_lng', 'receipt_lat', 'sign_time', 'sign_lng', 'sign_lat', 'ds']


,order_id,from_dipan_id,from_city_name,delivery_user_id,poi_lng,poi_lat,aoi_id,typecode,receipt_time,receipt_lng,receipt_lat,sign_time,sign_lng,sign_lat,ds
0,55be8cdf1270526231c9ba3387f51b54,c5ac5ba99801aa6b85ba473d9260512b,重庆市,df0b594618d1ba6f619e4e7dd034447c,8.899874e+06,-7.684936e+06,9c0f96ff01a71477334ef563001abc72,203ac3454d75e02ebb0a3c6f51d735e4,03-18 08:32:00,8.900992e+06,-7.686103e+06,03-18 14:33:00,NaN,NaN,318
1,21209805122ad6c7b39c203e77b8d9d8,3dbf3d4ed8dda395a48d49ada37ced6e,杭州市,868a64f65247a5f56490aee6c2eb96a8,1.040387e+07,-7.622290e+06,60a59ad10ca33ebff202c552fae6bfe9,203ac3454d75e02ebb0a3c6f51d735e4,03-18 08:26:00,1.040238e+07,-7.619631e+06,03-18 12:54:00,NaN,NaN,318
2,c00522c5a7072c8788fa9e069f3fe81a,8cde36f7816f76d731e2b3752be837ec,重庆市,be1b1aa5fcd0ee813fa8fe6bca78637e,8.910016e+06,-7.684918e+06,d8fc957cb109c1a0a241954a0c5aa5e2,203ac3454d75e02ebb0a3c6f51d735e4,03-18 07:53:00,8.908245e+06,-7.682531e+06,03-18 10:04:00,NaN,NaN,318
3,030c1387257ebfbbda45b7dea35ff1f8,12fc24c0aa6a27ec0cb3f8bb4d46ec36,重庆市,aba703cad68a631c96aa992d096cbe77,8.903061e+06,-7.676243e+06,5e9a1d1681ace9a86c1bab9c0d1924fc,203ac3454d75e02ebb0a3c6f51d735e4,03-18 07:58:00,8.906168e+06,-7.674608e+06,03-18 11:26:00,NaN,NaN,318
4,8f8a22238f13b4580785e3ad4e1643b6,281a1fe1179fa9099598fc6dcce365cc,上海市,dd648d4143daa61906cdb8c283aedb56,1.053426e+07,-7.479284e+06,fb4b17d3fb592d982bb38d29057baf15,203ac3454d75e02ebb0a3c6f51d735e4,03-18 16:56:00,1.053336e+07,-7.477580e+06,03-18 17:45:00,NaN,NaN,318


---

Columns in gps dump
- ds
- postman_id
- gps_time
- lat
- lng


---


Data in d5c ( original columns )

- order_id → Unique identifier for each package. Use: Links courier events to specific deliveries.

- from_dipan_id : Unique Identifier of the Originating Logistics Station
- city : City name (Shanghai, Hangzhou, etc.)
- delivery_user_id : eSSENTIALLY THE COURIER ID
- poi lng , lat : Customer's location
- aoi_id → Area of Interest ID (e.g., residential complex, office building). Use: Delivery zone granularity. numbers ~ 1700 Thousands of unique IDs per city.
- typecode : tells what kind of place it is , Limited set of standard categories (alpha numeric).
-

- courier_id → Unique identifier for the courier.

- 'lng', 'lat' : Coordinates of each stop in a 2D space (longitude/latitude). Use: Defines delivery/pickup locations.

- receipt_time : Warehourse receipt time
- receipt lat lng : warehouse coordinates
- sign_time : handover time
- sign_lat lng: not given , same as poi
- ds → Date of package delivery MMDD

---

For customer , we have poi lat lng and sign_time

for pickup at warehouse we have
receipt time , receipt lat lng

### 0.2 City Name Mapping + Datetime Parsing

Maps Chinese city names to English and parses the two timestamp columns.
The raw file stores times as `MM-DD HH:MM:SS` strings — "2021-" is prepended
to anchor them to the study year before converting to `datetime64`.

In [ ]:
CITY_MAPPING = {
    '上海市': 'Shanghai',
    '重庆市': 'Chongqing',
    '杭州市': 'Hangzhou',
}

wdata['from_city_en'] = wdata['from_city_name'].map(CITY_MAPPING)
wdata.rename(columns={'from_city_en': 'city'}, inplace=True)
wdata.drop(columns=['from_city_name'], inplace=True)

# Prepend year (data spans Mar–Apr 2021 as per paper)
wdata['receipt_time'] = pd.to_datetime(
    '2021-' + wdata['receipt_time'],
    format='%Y-%m-%d %H:%M:%S'
)
wdata['sign_time'] = pd.to_datetime(
    '2021-' + wdata['sign_time'],
    format='%Y-%m-%d %H:%M:%S'
)

unmapped = wdata['city'].isna().sum()
print(f"Unmapped city rows: {unmapped}")
print(f"Cities found: {wdata['city'].value_counts().to_dict()}")

Unmapped city rows: 0
Cities found: {'Hangzhou': 40774, 'Shanghai': 34736, 'Chongqing': 26673}


In [ ]:
wdata.head()

,order_id,from_dipan_id,delivery_user_id,poi_lng,poi_lat,aoi_id,typecode,receipt_time,receipt_lng,receipt_lat,sign_time,sign_lng,sign_lat,ds,city
0,55be8cdf1270526231c9ba3387f51b54,c5ac5ba99801aa6b85ba473d9260512b,df0b594618d1ba6f619e4e7dd034447c,8.899874e+06,-7.684936e+06,9c0f96ff01a71477334ef563001abc72,203ac3454d75e02ebb0a3c6f51d735e4,2021-03-18 08:32:00,8.900992e+06,-7.686103e+06,2021-03-18 14:33:00,NaN,NaN,318,Chongqing
1,21209805122ad6c7b39c203e77b8d9d8,3dbf3d4ed8dda395a48d49ada37ced6e,868a64f65247a5f56490aee6c2eb96a8,1.040387e+07,-7.622290e+06,60a59ad10ca33ebff202c552fae6bfe9,203ac3454d75e02ebb0a3c6f51d735e4,2021-03-18 08:26:00,1.040238e+07,-7.619631e+06,2021-03-18 12:54:00,NaN,NaN,318,Hangzhou
2,c00522c5a7072c8788fa9e069f3fe81a,8cde36f7816f76d731e2b3752be837ec,be1b1aa5fcd0ee813fa8fe6bca78637e,8.910016e+06,-7.684918e+06,d8fc957cb109c1a0a241954a0c5aa5e2,203ac3454d75e02ebb0a3c6f51d735e4,2021-03-18 07:53:00,8.908245e+06,-7.682531e+06,2021-03-18 10:04:00,NaN,NaN,318,Chongqing
3,030c1387257ebfbbda45b7dea35ff1f8,12fc24c0aa6a27ec0cb3f8bb4d46ec36,aba703cad68a631c96aa992d096cbe77,8.903061e+06,-7.676243e+06,5e9a1d1681ace9a86c1bab9c0d1924fc,203ac3454d75e02ebb0a3c6f51d735e4,2021-03-18 07:58:00,8.906168e+06,-7.674608e+06,2021-03-18 11:26:00,NaN,NaN,318,Chongqing
4,8f8a22238f13b4580785e3ad4e1643b6,281a1fe1179fa9099598fc6dcce365cc,dd648d4143daa61906cdb8c283aedb56,1.053426e+07,-7.479284e+06,fb4b17d3fb592d982bb38d29057baf15,203ac3454d75e02ebb0a3c6f51d735e4,2021-03-18 16:56:00,1.053336e+07,-7.477580e+06,2021-03-18 17:45:00,NaN,NaN,318,Shanghai


### 0.3 Split by City + Compute `eta_mins`

`eta_mins = (sign_time − receipt_time) / 60` in minutes — this becomes

In [ ]:
# storing individual city wise dataframes

shanghai_df  = wdata[wdata['city'] == 'Shanghai'].copy()
chongqing_df = wdata[wdata['city'] == 'Chongqing'].copy()
hangzhou_df  = wdata[wdata['city'] == 'Hangzhou'].copy()

for name, df in [('Shanghai', shanghai_df), ('Chongqing', chongqing_df), ('Hangzhou', hangzhou_df)]:
    df['eta_mins'] = (df['sign_time'] - df['receipt_time']).dt.total_seconds() / 60
    print(f"{name}: {len(df):,} rows | ETA range [{df['eta_mins'].min():.1f}, {df['eta_mins'].max():.1f}] min")

Shanghai: 34,736 rows | ETA range [-4446.0, 16772.0] min
Chongqing: 26,673 rows | ETA range [-6728.0, 31830.0] min
Hangzhou: 40,774 rows | ETA range [-21494.0, 24576.0] min


### 0.4 Remove Negative ETAs + ETA Distribution Diagnostics

Negative ETAs are data entry errors (sign_time < receipt_time).
Percentile diagnostics inform the tier thresholds used in the causal analysis:

- **Shanghai**: 90th pct ≈ 2.87 h, 99th pct ≈ 9.18 h — fast, reliable city
- **Hangzhou**: 90th pct ≈ 4.21 h — intermediate
- **Chongqing**: 90th pct ≈ 8.68 h, 99th pct ≈ 53.43 h — heavy right tail,
  topographic constraints (hilly terrain, river splits)

The KDE peak finder confirms Chongqing's mode at ≈ 2.12 h — unimodal
heavy-tailed (not bimodal; scipy peak detection can produce false peaks
on noisy tails, verified by visual inspection).

In [ ]:
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks

city_dfs = {
    'Shanghai':  shanghai_df,
    'Chongqing': chongqing_df,
    'Hangzhou':  hangzhou_df,
}

for city_name, df in city_dfs.items():
    neg = (df['eta_mins'] < 0).sum()
    print(f"\n--- {city_name} ---")
    print(f"  Negative ETAs: {neg:,}")

city_dfs_clean = {}
for city_name, df in city_dfs.items():
    cleaned = df[df['eta_mins'] >= 0].copy()
    city_dfs_clean[city_name] = cleaned

shanghai_df  = city_dfs_clean['Shanghai']
chongqing_df = city_dfs_clean['Chongqing']
hangzhou_df  = city_dfs_clean['Hangzhou']


--- Shanghai ---
  Negative ETAs: 1

--- Chongqing ---
  Negative ETAs: 3

--- Hangzhou ---
  Negative ETAs: 30


In [ ]:
# ── ETA percentile diagnostics (no plots) ────────────────────────────────────
PERCENTILES = [10, 25, 50, 75, 90, 95, 99]

for city_name, df in city_dfs_clean.items():
    eta_hours = df['eta_mins'] / 60
    eta_clean = eta_hours.dropna()

    print(f"\n--- {city_name} ---")

    # Percentiles
    for p in PERCENTILES:
        print(f"  {p:>2}th pct: {np.percentile(eta_clean, p):.2f} h")

    # KDE peak finding (text output only)
    if len(eta_clean) >= 2:
        x_max = max(eta_clean.quantile(0.99) * 1.5, 10)
        eta_kde = eta_clean[eta_clean < x_max * 1.1]
        kde = gaussian_kde(eta_kde)
        xs  = np.linspace(0, x_max, 1000)
        ys  = kde(xs)
        peaks, _ = find_peaks(ys, height=ys.max() * 0.05)
        if len(peaks):
            locs = [f"{xs[p]:.2f}h" for p in peaks]
            print(f"  KDE peaks: {locs}")
        else:
            print("  KDE peaks: none found")


--- Shanghai ---
  10th pct: 0.38 h
  25th pct: 0.68 h
  50th pct: 1.17 h
  75th pct: 1.90 h
  90th pct: 2.87 h
  95th pct: 3.72 h
  99th pct: 9.18 h
  KDE peaks: ['0.77h']

--- Chongqing ---
  10th pct: 0.60 h
  25th pct: 1.20 h
  50th pct: 2.28 h
  75th pct: 3.98 h
  90th pct: 8.68 h
  95th pct: 12.08 h
  99th pct: 53.43 h
  KDE peaks: ['1.68h', '8.75h']

--- Hangzhou ---
  10th pct: 0.48 h
  25th pct: 0.87 h
  50th pct: 1.52 h
  75th pct: 2.58 h
  90th pct: 4.21 h
  95th pct: 6.68 h
  99th pct: 13.61 h
  KDE peaks: ['0.94h']


### 0.5 Save Per-City Delivery CSVs

Outputs consumed by Stage 1's `load_delivery()`:
`shanghai_data.csv`, `chongqing_data.csv`, `hangzhou_data.csv`

In [ ]:
shanghai_df.to_csv( CITY_DIR + '/shanghai_data.csv',  index=False)
chongqing_df.to_csv(CITY_DIR + '/chongqing_data.csv', index=False)
hangzhou_df.to_csv( CITY_DIR + '/hangzhou_data.csv',  index=False)

for name, df in [('Shanghai', shanghai_df), ('Hangzhou', hangzhou_df), ('Chongqing', chongqing_df)]:
    print(f"  Saved {name}: {len(df):,} rows → {CITY_DIR}/{name.lower()}_data.csv")

  Saved Shanghai: 34,735 rows → /content/drive/MyDrive/ml/PROCESSED/matched/city_divided/shanghai_data.csv
  Saved Hangzhou: 40,744 rows → /content/drive/MyDrive/ml/PROCESSED/matched/city_divided/hangzhou_data.csv
  Saved Chongqing: 26,670 rows → /content/drive/MyDrive/ml/PROCESSED/matched/city_divided/chongqing_data.csv


### 0.6 Identify Dropped Courier IDs

Couriers whose **only** orders had negative ETAs are absent from the cleaned
DataFrames. Their GPS trajectories must also be excluded from the GPS parquet
to keep the delivery and GPS tables consistent.

Summary (study results):  
- Shanghai: 1 dropped courier ID  
- Chongqing: 0  
- Hangzhou: 9  
- GPS rows removed from parquet: ~50,807 / 16,042,735

In [ ]:
# Rebuild original city slices (pre-filter) for ID comparison
for city_name, df in city_dfs.items():
    df['eta_mins'] = (df['sign_time'] - df['receipt_time']).dt.total_seconds() / 60

original_ids = {
    'Shanghai':  set(city_dfs['Shanghai']['delivery_user_id'].unique()),
    'Chongqing': set(city_dfs['Chongqing']['delivery_user_id'].unique()),
    'Hangzhou':  set(city_dfs['Hangzhou']['delivery_user_id'].unique()),
}
filtered_ids = {
    'Shanghai':  set(shanghai_df['delivery_user_id'].unique()),
    'Chongqing': set(chongqing_df['delivery_user_id'].unique()),
    'Hangzhou':  set(hangzhou_df['delivery_user_id'].unique()),
}

dropped_ids = {}
all_dropped = set()
for city in ['Shanghai', 'Chongqing', 'Hangzhou']:
    dropped = original_ids[city] - filtered_ids[city]
    dropped_ids[city] = dropped
    all_dropped |= dropped
    print(f"  {city}: {len(dropped)} dropped courier ID(s)")

print(f"\n  Total unique dropped IDs: {len(all_dropped)}")

  Shanghai: 1 dropped courier ID(s)
  Chongqing: 0 dropped courier ID(s)
  Hangzhou: 9 dropped courier ID(s)

  Total unique dropped IDs: 10


### 0.7 Filter GPS Parquet + Split by City

Removes dropped courier rows from the full GPS parquet, then splits by city
using the delivery CSVs' `delivery_user_id` sets.

Output files consumed by Stage 1's `load_gps_window()`:
`{city}_delivery_data.parquet`

In [ ]:
GPS_PARQUET = BASE_RAW + '/tj_processed_join_filtered.parquet'
GPS_FILTERED_OUT = BASE_RAW + '/tj_process_negative_removed.parquet'

parquet_df = pd.read_parquet(GPS_PARQUET)
print(f"GPS parquet shape (before filter): {parquet_df.shape}")

# Remove rows for dropped couriers
parquet_df_filtered = parquet_df[~parquet_df['postman_id'].isin(all_dropped)].copy()
print(f"GPS parquet shape (after filter):  {parquet_df_filtered.shape}")
print(f"Rows removed: {len(parquet_df) - len(parquet_df_filtered):,}")

parquet_df_filtered.to_parquet(GPS_FILTERED_OUT, index=False)
print(f"\nSaved filtered GPS parquet → {GPS_FILTERED_OUT}")

GPS parquet shape (before filter): (16042735, 5)
GPS parquet shape (after filter):  (15991938, 5)
Rows removed: 50,797

Saved filtered GPS parquet → /content/drive/MyDrive/ml/PROCESSED/matched/tj_process_negative_removed.parquet


In [ ]:
# ── Split GPS parquet by city using delivery_user_id sets ─────────────────────
city_delivery_ids = {
    'shanghai':  set(shanghai_df['delivery_user_id'].unique()),
    'chongqing': set(chongqing_df['delivery_user_id'].unique()),
    'hangzhou':  set(hangzhou_df['delivery_user_id'].unique()),
}

for slug, ids in city_delivery_ids.items():
    city_gps = parquet_df_filtered[
        parquet_df_filtered['postman_id'].isin(ids)
    ].copy()

    out_path = f"{CITY_DIR}/{slug}_delivery_data.parquet"
    city_gps.to_parquet(out_path, index=False)
    print(f"  {slug.capitalize()} GPS: {city_gps.shape[0]:,} rows → {out_path}")

print("\n✅ Stage 0 complete — all city files ready for Stage 1.")

  Shanghai GPS: 3,592,545 rows → /content/drive/MyDrive/ml/PROCESSED/matched/city_divided/shanghai_delivery_data.parquet
  Chongqing GPS: 4,914,746 rows → /content/drive/MyDrive/ml/PROCESSED/matched/city_divided/chongqing_delivery_data.parquet
  Hangzhou GPS: 7,484,647 rows → /content/drive/MyDrive/ml/PROCESSED/matched/city_divided/hangzhou_delivery_data.parquet

✅ Stage 0 complete — all city files ready for Stage 1.


---

## Stage 1 · Feature Engineering

All cells below are the feature engineering pipeline run per city .
Inputs are the per-city CSV and GPS parquet files produced in Stage 0 above.

In [ ]:
!pip install duckdb polars pyarrow --quiet

In [ ]:
import polars as pl
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import date

pl.Config.set_tbl_rows(10)
print("Polars:", pl.__version__, " | DuckDB:", duckdb.__version__)

Polars: 1.35.2  | DuckDB: 1.3.2


## Pipeline Overview

```
load_delivery()
add_euclidean_distance()
add_batch_features()
add_workload_nonlinear()
compute_speed_percentile()
compute_trajectory_features()
courier_snapshot()
filter_stale_gps()
add_operational_and_distance_features()
add_gps_missingness_flag()
add_temporal_features()
add_typecode_encoding()
add_spatial_congestion_v2()
add_weather()
```

Each section below is a reusable function. The **Run All Cities** section
loops over all three cities and saves per-city parquets plus a combined parquet.

## 0. Install & Imports

### Delivery level feature engineering

## 1. Configuration

Edit paths and parameters below before running.

- **`CITY_CONFIGS`** — add/remove cities here; each entry carries its own file names,
  weather CSV path, and holiday list so the pipeline is fully self-contained per city.
- **`PRE_MIN`** — GPS look-back window (minutes before `receipt_time`).
- **`MAX_GAP`** — GPS matches older than this are invalidated as stale.
- **`GRID_SIZE`** — spatial grid cell size in affine coordinate units (≈ 500 m).
- **`WORKLOAD_CAP`** — cap applied in `add_workload_nonlinear` to suppress extreme outliers.
- **`ROLLING_DAYS`** — reserved for future rolling-window features.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE         = "/content/drive/MyDrive/ml/PROCESSED/matched/city_divided/"
WEATHER_BASE = "/content/drive/MyDrive/ml/weather-outputs/"
OUTPUT_DIR   = "/content/drive/MyDrive/ml/CORRECTEDv3/"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ── Pipeline hyper-parameters ─────────────────────────────────────────────────
PRE_MIN      = 15     # minutes of GPS history before receipt_time
MAX_GAP      = 30     # GPS staleness threshold (minutes)
GRID_SIZE    = 500    # spatial grid cell size (affine coordinate units ≈ 500 m)
WORKLOAD_CAP = 20     # workload cap for non-linear features
ROLLING_DAYS = 7      # reserved for future rolling-window features

# ── City registry ─────────────────────────────────────────────────────────────
# Each entry:
#   city_en       : human-readable name (must match CITY_IN_QUES in weather CSVs)
#   delivery_file : CSV filename under BASE
#   gps_file      : parquet filename under BASE
#   weather_file  : CSV filename under WEATHER_BASE
#   holidays      : list of 'YYYY-MM-DD' strings (Qingming Festival 2021)
#   holiday_eve   : day before first holiday
CITY_CONFIGS = [
    {
        "city_en"      : "Shanghai",
        "delivery_file": "shanghai_data.csv",
        "gps_file"     : "shanghai_delivery_data.parquet",
        "weather_file" : "shanghai_wsi_mar17_apr20_2021.csv",
        "holidays"     : ["2021-04-03", "2021-04-04", "2021-04-05"],
        "holiday_eve"  : "2021-04-02",
    },
    {
        "city_en"      : "Hangzhou",
        "delivery_file": "hangzhou_data.csv",
        "gps_file"     : "hangzhou_delivery_data.parquet",
        "weather_file" : "hangzhou_wsi_mar17_apr20_2021.csv",
        "holidays"     : ["2021-04-03", "2021-04-04", "2021-04-05"],
        "holiday_eve"  : "2021-04-02",
    },
    {
        "city_en"      : "Chongqing",
        "delivery_file": "chongqing_data.csv",
        "gps_file"     : "chongqing_delivery_data.parquet",
        "weather_file" : "chongqing_wsi_mar17_apr20_2021.csv",
        "holidays"     : ["2021-04-03", "2021-04-04", "2021-04-05"],
        "holiday_eve"  : "2021-04-02",
    },
]



In [ ]:
# ── Selected features for downstream causal / RL stages ──────────────────────
SELECTED_FEATURES = [
    "order_id", "city", "batch_id",
    "batch_size", "batch_rank_dispatch", "batch_rank_actual",
    "delivery_sequence_daily",
    "speed_mean", "distance_travelled", "gps_points", "gps_gap_min",
    "is_trajectory_available",
    "pickup_destination_distance",
    "hour_sin", "hour_cos", "day_sin", "day_cos",
    "is_holiday", "is_holiday_eve", "is_weekend",
    "WSI", "precipitation", "temperature_2m", "windspeed_10m",
    "typecode_grouped_type 1", "typecode_grouped_type 2",
    "typecode_grouped_type 3", "typecode_cb",
    "spatial_congestion_daily", "spatial_congestion_norm",
    "courier_local_load",
    "eta_mins",
]

## 2. Pipeline Functions

All transformation logic lives here.  Functions are **stateless** — they receive a
dataframe (and sometimes a DuckDB connection / path) and return a new dataframe.
No global mutation.

---

# Engineered Feature buckets

* ### Structural features
-

---
* ### Operational state features
-

---
* ### Environmental features
-

---
* ### Temporal Control Variables
-
---

* ### Diagonistic / Post Adhoc features
-

### 2.2 `load_gps_window` — Fetch relevant GPS records via DuckDB

Uses DuckDB push-down filtering to avoid loading the full GPS parquet into memory.
Only couriers active during the delivery window (±`pre_min` minutes) are pulled.

### 2.3 `courier_snapshot` + `filter_stale_gps` — Last known position

ASOF backward join attaches the courier's most recent GPS ping at or before each
`receipt_time`. Pings older than `max_gap` minutes are then nulled out — stale GPS
is operationally unreliable and structurally different from observed zero-speed
movement (indoor delivery, urban canyon signal loss, device quality).

### 2.4 `compute_speed_percentile` — 99th-percentile speed cap

A robust speed ceiling computed from the full GPS parquet.  
Filters: `dt > 5 s` (removes GPS duplicates), `dt < 600 s` (removes long gaps),
`dist < 10000` (removes privacy-related coordinate jumps).

### 2.5 `compute_trajectory_features` — 15-min pre-delivery GPS window

Aggregates GPS segments in `[receipt_time − pre_min, receipt_time]` per order.
Speed filtering (per segment): removes GPS duplicates (`dt ≤ 1 s`), privacy jumps
(`dist > 5000`), and physically impossible speeds (above 99th-percentile cap).

### 2.7 `add_batch_features` — Batch size, dispatch rank, actual rank, batch ID


| Column | Definition | Causal role |
|---|---|---|
| `batch_rank_dispatch` | Order within simultaneous push (sorted by `order_id`) — deterministic, reproducible | Pre-delivery operational context |
| `batch_rank_actual` | Rank by `sign_time` within the batch — post-hoc delivery sequence | Observed service order; do **not** use as a predictor (data leakage from target) |
| `batch_id` | `delivery_user_id + "__" + epoch(receipt_time)` — explicit batch key | Enables batch-level aggregation without re-grouping |

`batch_rank_actual` is stored for analysis only. Use `batch_rank_dispatch` in models.

### 2.1 `load_delivery` — Load, normalise, and filter the delivery CSV

**ETA filter :** rows with `eta_mins ≤ 0` or `≥ 1440` are dropped.
- `eta ≤ 0` — data entry errors; a negative delivery time is physically impossible.
- `eta ≥ 1440` — exceeds 24 h; almost certainly warehousing delays or GPS silence
  rather than active last-mile delivery (confirmed via percentile based ETA long-tail analysis).

### 2.12 `add_typecode_encoding` — OHE grouping + frequency encoding

Dual encoding for the package type field.  
(a) **OHE grouping** — top-N codes → named groups; rest → 'rare'. Binary columns
serve as root nodes in the causal graph.  
(b) **Frequency encoding** — dense ordinal signal for tree models and bandit policies.

### 2.13 `add_spatial_congestion` — Grid-based SCI + courier local load

**v2 change:** `spatial_congestion_daily` groups by `(grid_x, grid_y, receipt_date, time_window)`
instead of just `(grid_x, grid_y, time_window)`. This prevents cross-day leakage — the
same hour-of-day cell on different dates now has separate counts.

Spatial Congestion Index (SCI) adapted from Ke et al. (SIGKDD 2017).

### 2.14 `add_weather` — Temporal ASOF join with hourly weather

Joins hourly Open-Meteo observations to deliveries via ASOF backward join.
Each `receipt_time` is matched to the most recent prior weather record.
The WSI (Weather Severity Index) is a weighted composite of precipitation (0.30),
wind speed (0.25), weather code severity (0.25), and temperature deviation (0.20)
— defended using the OECD/JRC composite index framework (Nardo et al., 2008).

### 2.16 `final_sanity_check` — Validation gates

Four-gate validation. Raises `AssertionError` on failure so the pipeline stops
rather than silently saving a corrupt features file. Updated in v2 to check
`workload_causal` and the new `batch_id` / `batch_rank_dispatch` columns.

In [ ]:
# def final_sanity_check(features: pl.DataFrame, city_en: str) -> None:
#     """
#     Four-gate validation. Raises AssertionError on failure so the pipeline
#     stops rather than silently saving a corrupt features file.
#     """
#     print(f"\n{'─'*55}")
#     print(f"  Sanity check: {city_en}")
#     print(f"{'─'*55}")

#     # 1. No null targets
#     null_eta = features["eta_mins"].null_count()
#     assert null_eta == 0, f"FAIL: {null_eta} null eta_mins values!"
#     print(f"  ✓ eta_mins null count: 0")

#     # 2. Row uniqueness
#     assert features.height == features["order_id"].n_unique(), \
#         "FAIL: duplicate order_id rows detected!"
#     print(f"  ✓ Row count == unique order_id count: {features.height:,}")

#     # 3. No accidental join duplicates
#     dup_cols = [c for c in features.columns if c.endswith("_right")]
#     assert not dup_cols, f"FAIL: duplicate columns found: {dup_cols}"
#     print(f"  ✓ No duplicate columns")

#     # 4. Workload causal validity
#     assert features["workload_causal"].min() >= 0, "FAIL: negative workload_causal!"
#     print(f"  ✓ workload_causal ≥ 0")

#     # 5. Key feature null counts
#     key_cols = [
#         "workload_causal", "batch_rank_dispatch", "batch_rank_actual",
#         "workload_capped", "batch_id",
#         "hour_sin", "WSI", "is_trajectory_available",
#         "spatial_congestion_daily",
#     ]
#     for col in key_cols:
#         if col in features.columns:
#             n = features[col].null_count()
#             status = "✓" if n == 0 else "⚠"
#             print(f"  {status} {col} nulls: {n}")

#     print(f"  Total columns: {len(features.columns)}")
#     print(f"  Total rows:    {features.height:,}")
#     print(f"{'─'*55}\n")

## 3. EDA / Diagnostic Functions

Quick visual diagnostics per city. Call after `build_city_features()` or from
the cross-city comparison section.

In [ ]:
# def plot_speed_ccdf(con, gps_path: str, city_en: str) -> None:
#     """
#     Log-log CCDF of courier speed.
#     A heavy right tail confirms the trajectory captures real inter-zone movement.
#     A sharp drop-off at the far end = rare cross-city reallocation events.
#     """
#     speed_df = con.execute(f"""
#         WITH motion AS (
#             SELECT
#                 SQRT(POWER(lat - LAG(lat) OVER w, 2) +
#                      POWER(lng - LAG(lng) OVER w, 2)) AS dist,
#                 EXTRACT(EPOCH FROM (gps_time - LAG(gps_time) OVER w)) AS dt
#             FROM parquet_scan('{gps_path}')
#             WINDOW w AS (PARTITION BY postman_id ORDER BY gps_time)
#         )
#         SELECT dist / dt AS speed
#         FROM motion
#         WHERE dt > 0 AND dist IS NOT NULL AND dist / dt IS NOT NULL
#     """).pl().sort("speed")

#     n = speed_df.height
#     ccdf = speed_df.with_row_index("rank").with_columns(
#         (1 - pl.col("rank") / n).alias("ccdf")
#     )
#     s = ccdf["speed"].to_numpy()
#     p = ccdf["ccdf"].to_numpy()
#     mask = (s > 0) & (p > 0)

#     plt.figure(figsize=(6, 4))
#     plt.loglog(s[mask], p[mask])
#     plt.xlabel("Speed (log scale)")
#     plt.ylabel("P(Speed ≥ v) (log scale)")
#     plt.title(f"Courier Speed CCDF — {city_en}")
#     plt.grid(True, which="both", ls="--", alpha=0.5)
#     plt.tight_layout()
#     plt.show()


# def plot_workload_saturation(features: pl.DataFrame, city_en: str) -> None:
#     """
#     Bar chart: mean ETA vs workload_causal (clipped at 20).
#     Illustrates M/G/1 saturation curve — delay explodes near ρ→1.
#     """
#     plot_df = (
#         features
#         .with_columns(pl.col("workload_causal").clip(upper_bound=20))
#         .group_by("workload_causal")
#         .agg(pl.mean("eta_mins").alias("mean_eta"), pl.len().alias("n"))
#         .sort("workload_causal")
#         .to_pandas()
#     )
#     plt.figure(figsize=(11, 4))
#     sns.barplot(x="workload_causal", y="mean_eta",
#                 data=plot_df, palette="viridis",
#                 hue="workload_causal", legend=False)
#     plt.axvline(x=14.5, color="red", linestyle="--", alpha=0.7, label="Saturation threshold")
#     plt.title(f"Workload Saturation Curve — {city_en}")
#     plt.xlabel("workload_causal (clipped at 20)")
#     plt.ylabel("Mean ETA (mins)")
#     plt.legend()
#     plt.tight_layout()
#     plt.show()


# def plot_batch_rank_eta(features: pl.DataFrame, city_en: str) -> None:
#     """
#     Line chart: mean ETA vs batch_rank_dispatch (clipped at 15).
#     Monotonic rise validates the late_batch binary feature.
#     Uses dispatch rank only — no leakage from batch_rank_actual.
#     """
#     plot_df = (
#         features
#         .with_columns(pl.col("batch_rank_dispatch").clip(upper_bound=15))
#         .group_by("batch_rank_dispatch")
#         .agg(pl.mean("eta_mins").alias("mean_eta"), pl.len().alias("n"))
#         .sort("batch_rank_dispatch")
#         .to_pandas()
#     )
#     plt.figure(figsize=(9, 4))
#     plt.plot(plot_df["batch_rank_dispatch"], plot_df["mean_eta"], marker="o")
#     plt.fill_between(plot_df["batch_rank_dispatch"],
#                      plot_df["mean_eta"] * 0.9, plot_df["mean_eta"] * 1.1,
#                      alpha=0.15)
#     plt.axvline(x=8, color="orange", linestyle="--", alpha=0.7, label="late_batch threshold")
#     plt.title(f"Batch Rank (Dispatch) vs Mean ETA — {city_en}")
#     plt.xlabel("batch_rank_dispatch (clipped at 15)")
#     plt.ylabel("Mean ETA (mins)")
#     plt.legend()
#     plt.tight_layout()
#     plt.show()


# def plot_eta_distribution(features: pl.DataFrame, city_en: str, clip_pct: float = 0.95) -> None:
#     """
#     KDE of eta_mins clipped at the 95th percentile to suppress extreme outliers.
#     """
#     cap = features["eta_mins"].quantile(clip_pct)
#     data = features.filter(pl.col("eta_mins") <= cap)["eta_mins"].to_numpy()
#     plt.figure(figsize=(8, 4))
#     sns.kdeplot(data, fill=True)
#     plt.axvline(data.mean(), color="red",  linestyle="--", label=f"Mean {data.mean():.0f}")
#     plt.axvline(float(np.median(data)), color="green", linestyle="--",
#                 label=f"Median {float(np.median(data)):.0f}")
#     plt.title(f"ETA Distribution (≤P{int(clip_pct*100)}) — {city_en}")
#     plt.xlabel("ETA (mins)")
#     plt.ylabel("Density")
#     plt.legend()
#     plt.tight_layout()
#     plt.show()


# def print_correlation_table(features: pl.DataFrame, city_en: str) -> None:
#     """
#     Key Pearson correlations with eta_mins.
#     Uses v2 column names (workload_causal, batch_rank_dispatch).
#     """
#     targets = [
#         "workload_causal", "workload_capped",
#         "batch_rank_dispatch", "batch_rank_capped",
#         "speed_mean", "gps_gap_min",
#         "pickup_destination_distance", "spatial_congestion_norm",
#         "WSI", "hour_sin", "delivery_sequence_daily",
#     ]
#     rows = []
#     for col in targets:
#         if col in features.columns:
#             rho = features.select(pl.corr(col, "eta_mins")).item()
#             rows.append((col, f"{rho:+.4f}"))
#     print(f"\nPearson ρ with eta_mins — {city_en}")
#     print(f"{'Feature':<38} {'ρ':>8}")
#     print("─" * 48)
#     for feat, rho in sorted(rows, key=lambda x: abs(float(x[1])), reverse=True):
#         print(f"  {feat:<36} {rho:>8}")

Exponentially Weighted Moving Average of a courier's past delivery times. It represents the 'historical performance' of the courier; a low value suggests an historically fast courier.

## 4. Master Pipeline Function

`build_city_features` orchestrates all steps for a single city config dict.
Pass `run_eda_plots=False` for silent batch runs.

gps_points: The raw count of GPS pings recorded in the 15-minute window before the order was received

In [ ]:
# just loads necessary columns for delivery operations
def load_delivery(csv_path: str) -> pl.DataFrame:
    # Applies ETA validity filter  -- in between 0 mins and 1 day
    df = (pl.read_csv(csv_path)
          .with_columns([pl.col("receipt_time").str.to_datetime("%Y-%m-%d %H:%M:%S"),
                         pl.col("sign_time").str.to_datetime("%Y-%m-%d %H:%M:%S")])
                .drop(["sign_lat", "sign_lng"])
                .with_columns(pl.col("delivery_user_id").cast(pl.Utf8)))

    df = df.filter((pl.col("eta_mins" ) > 0) & (pl.col("eta_mins") < 1440))
    return df.sort(["delivery_user_id", "from_dipan_id", "ds", "receipt_time"])  # sorting direction is imp

### 2.11 `add_gps_missingness_flag` — Structural fill + indicator

Encodes GPS absence as a first-class operational state.  
Strategy: structural fill (0) + binary indicator, **not** statistical imputation.
Missing GPS ≠ stationary courier — it correlates with indoor deliveries, dense urban
canyons, and device quality (structural missingness, not MCAR). (MCAR) data occurs when the probability of missingness is independent of any observed or unobserved data

In [ ]:
def courier_snapshot(delivery: pl.DataFrame, gps_df: pl.DataFrame) -> pl.DataFrame:
    # ASOF backward join: attach most recent GPS point
    # Renaming postman_id to match delivery_user_id for 'by' compatibility
    gps_renamed = gps_df.rename({"postman_id": "delivery_user_id"})

    state = delivery.join_asof(
        gps_renamed,
        left_on="receipt_time",
        right_on="gps_time",
        by="delivery_user_id",
        strategy="backward"
    ).rename({"lat": "last_x", "lng": "last_y", "gps_time": "last_gps_time"})

    return state.with_columns(
        (pl.col("receipt_time") - pl.col("last_gps_time")).dt.total_minutes()
        .alias("gps_gap_min")
    )

def filter_stale_gps(state: pl.DataFrame, max_gap: int = 30) -> pl.DataFrame:
    # Nullify GPS data if the last ping is older than max_gap minutes
    state = state.with_columns(
        pl.when(pl.col("gps_gap_min") <= max_gap)
        .then(pl.col("gps_gap_min"))
        .otherwise(None).alias("gps_gap_min")
    )
    return state.with_columns([
        pl.when(pl.col("gps_gap_min").is_null()).then(None).otherwise(pl.col("last_x")).alias("last_x"),
        pl.when(pl.col("gps_gap_min").is_null()).then(None).otherwise(pl.col("last_y")).alias("last_y")
    ])

In [ ]:
# Straightline euclidean distance on affine space

def add_euclidean_distance(delivery: pl.DataFrame) -> pl.DataFrame:
    return delivery.with_columns((
        (pl.col("receipt_lng") - pl.col("poi_lng")).pow(2) + (pl.col("receipt_lat") - pl.col("poi_lat")).pow(2)).sqrt()
            .alias("pickup_destination_distance"))

In [ ]:
import polars as pl

# batching features - that are available at delivery level and can affect individual deliveries

# concept of batching :  starts from same warehouse = same from_dipan_id ,
#                       same courier id , and receipt time is same at same ds (MM/YY)
#                       with a 5-minute window for receipt_time

def add_batch_features(delivery: pl.DataFrame) -> pl.DataFrame:
    # 1. Truncate receipt_time to a 5-minute window for batch definition
    delivery = delivery.with_columns(
        pl.col("receipt_time").dt.truncate("5m").alias("receipt_time_window")
    )

    # Define new batch keys using the 5-minute window
    batch_grouping_keys = ["delivery_user_id", "from_dipan_id", "ds", "receipt_time_window"]

    # Sort by batch_grouping_keys and original receipt_time for deterministic batch ordering within the window
    # and then by order_id. This ensures stable ranking if multiple orders fall into the same 5-min window.
    delivery = delivery.sort(batch_grouping_keys + ["receipt_time", "order_id"])

    # Calculate batch_size
    delivery = delivery.with_columns(
        pl.len().over(batch_grouping_keys).alias("batch_size")
    )

    # Calculate same_aoi_count: count occurrences of each aoi_id within each batch (intermediate calculation)
    delivery = delivery.with_columns(
        pl.col("aoi_id").count().over(batch_grouping_keys + ["aoi_id"]).alias("_same_aoi_count")
    )
    # Calculate isolated_delivery: 1 if same_aoi_count is 1, else 0
    delivery = delivery.with_columns(
        (pl.col("_same_aoi_count") == 1).cast(pl.Int8).alias("isolated_delivery")
    )
    # Calculate same_aoi_share_in_batch: _same_aoi_count normalized by batch_size
    delivery = delivery.with_columns(
        (pl.col("_same_aoi_count") / pl.col("batch_size")).alias("same_aoi_share_in_batch")
    )

    # Pre-delivery feature: batch_rank_dispatch (order within the batch, based on sort order)
    delivery = delivery.with_columns(
        pl.int_range(0, pl.len()).over(batch_grouping_keys).alias("batch_rank_dispatch")
    )

    # Post-delivery feature: batch_rank_actual (rank by sign_time within the batch)
    # This is for validation only and should not be used as a predictor.
    delivery = delivery.with_columns(
        (pl.col("sign_time").rank("ordinal").over(batch_grouping_keys) - 1).alias("batch_rank_actual")
    )

    # Unique ID for each batch, reflecting the 5-minute window
    delivery = delivery.with_columns((
        pl.col("delivery_user_id") + "__" +
        pl.col("from_dipan_id") + "__" +
        pl.col("ds").cast(pl.Utf8) + "__" +
        pl.col("receipt_time_window").dt.epoch("s").cast(pl.Utf8)
    ).alias("batch_id"))

    # Cumulative centroid and distance
    # These features use 'batch_id' which is now defined based on the 5-min window.
    # They capture within-batch efficiency and spatial compactness tracking.
    delivery = delivery.with_columns([
        (pl.col("poi_lng").cum_sum().over("batch_id") / (pl.col("batch_rank_dispatch") + 1)).alias("batch_cum_centroid_lng"),
        (pl.col("poi_lat").cum_sum().over("batch_id") / (pl.col("batch_rank_dispatch") + 1)).alias("batch_cum_centroid_lat")
    ])

    delivery = delivery.with_columns(
        ((pl.col("poi_lng") - pl.col("batch_cum_centroid_lng")).pow(2) +
         (pl.col("poi_lat") - pl.col("batch_cum_centroid_lat")).pow(2)).sqrt().alias("distance_to_batch_centroid")
    ).drop(["batch_cum_centroid_lng", "batch_cum_centroid_lat"])

    # Drop the temporary '_same_aoi_count' and 'receipt_time_window' columns
    delivery = delivery.drop(["_same_aoi_count", "receipt_time_window"])

    return delivery

In [ ]:
# for contextual bandits -- should not be included as predictor

def add_batch_duration_features(df: pl.DataFrame) -> pl.DataFrame:
    """
    Calculates the duration between consecutive deliveries within a batch.
    Sorted by batch_rank_actual to capture chronological delivery intervals.
    """
    # Ensure we are sorted by batch and the actual sequence of delivery
    df = df.sort(["batch_id", "batch_rank_actual"])

    # Calculate time since previous delivery in the same batch (in minutes)
    df = df.with_columns(
        ((pl.col("sign_time") - pl.col("sign_time").shift(1).over("batch_id"))
         .dt.total_seconds() / 60.0).alias("last_delivery_duration")
    )

    # Fill the first delivery of each batch (which has no 'previous') with 0 or null
    # Here we fill with 0 to indicate it's the start of the sequence
    return df.with_columns(pl.col("last_delivery_duration").fill_null(0.0))


    print("Updating city results with last_delivery_duration...")

for city in city_results_v4:
    city_results_v4[city] = add_batch_duration_features(city_results_v4[city])
    print(f"  ✓ Updated {city}")

# Re-combine the master dataset to include the new column
combined_v4 = pl.concat(list(city_results_v4.values()), how="diagonal")

# Quick verification of the new feature for a sample batch
sample_batch = combined_v4.filter(pl.col("batch_size") > 2).get_column("batch_id")[0]
display(combined_v4.filter(pl.col("batch_id") == sample_batch)
        .select(["batch_id", "batch_rank_actual", "sign_time", "last_delivery_duration"])
        .sort("batch_rank_actual"))

NameError: name 'city_results_v4' is not defined

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution across the entire combined dataset
plt.figure(figsize=(10, 6))
sns.histplot(combined_v4['last_delivery_duration'].to_numpy(), bins=50, kde=True, color='teal')

plt.title('Global Distribution of Last Delivery Duration')
plt.xlabel('Duration from Previous Delivery (mins)')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Statistical markers
mean_val = combined_v4['last_delivery_duration'].mean()
median_val = combined_v4['last_delivery_duration'].median()
plt.axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.2f}')
plt.axvline(median_val, color='orange', linestyle='-', label=f'Median: {median_val:.2f}')

plt.legend()
plt.show()

In [ ]:
import numpy as np
import polars as pl

# 1. Calculate the 99th percentile for clipping
cap_99 = combined_v4['last_delivery_duration'].quantile(0.99)
print(f'99th Percentile: {cap_99:.2f} minutes')

# 2. Apply clipping and create the binary flag for first-in-batch deliveries
combined_v4 = combined_v4.with_columns([
    pl.col('last_delivery_duration').clip(upper_bound=cap_99).alias('last_delivery_duration_clipped'),
    pl.when(pl.col('last_delivery_duration') == 0.0).then(1).otherwise(0).alias('is_first_in_batch')
])

# 3. Update the individual city results in the dictionary to maintain consistency
for city in city_results_v4:
    city_results_v4[city] = city_results_v4[city].with_columns([
        pl.col('last_delivery_duration').clip(upper_bound=cap_99).alias('last_delivery_duration_clipped'),
        pl.when(pl.col('last_delivery_duration') == 0.0).then(1).otherwise(0).alias('is_first_in_batch')
    ])

# Visualize the clipped distribution
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(10, 6))
sns.histplot(combined_v4['last_delivery_duration_clipped'].to_numpy(), bins=50, kde=True, color='olive')
plt.title(f'Clipped Distribution of Last Delivery Duration (Cap: {cap_99:.2f} min)')
plt.xlabel('Duration (mins)')
plt.ylabel('Frequency')
plt.show()

print(f"Binary flag 'is_first_in_batch' created. Sample distribution:")
print(combined_v4['is_first_in_batch'].value_counts())

In [ ]:
import polars as pl

# Drop the raw 'last_delivery_duration' column to avoid multicollinearity
combined_v4 = combined_v4.drop('last_delivery_duration')

# Also update the city-specific results dictionary
for city in city_results_v4:
    if 'last_delivery_duration' in city_results_v4[city].columns:
        city_results_v4[city] = city_results_v4[city].drop('last_delivery_duration')

print("✓ Original 'last_delivery_duration' dropped from combined_v4 and city_results_v4.")
print(f"Remaining columns: {[c for c in combined_v4.columns if 'last_delivery' in c]}")

In [ ]:
# Movement details - gps

def compute_speed_percentile(con, gps_path, percentile=0.99):
    # Robust speed ceiling : By calculating a 99th percentile speed cap, we ensure that the subsequent trajectory features aren't skewed by physically impossible speeds.
    speed_q = con.execute(f"SELECT quantile_cont(dist/dt, {percentile}) FROM (SELECT SQRT(POWER(lat-LAG(lat) OVER w,2)+POWER(lng-LAG(lng) OVER w,2)) AS dist, EXTRACT(EPOCH FROM (gps_time-LAG(gps_time) OVER w)) AS dt FROM parquet_scan('{gps_path}') WINDOW w AS (PARTITION BY postman_id ORDER BY gps_time)) WHERE dt>5 AND dt<600 AND dist<10000").fetchone()[0]
    return speed_q

def compute_trajectory_features(con, gps_path, speed_cap, pre_min=15):
    # 15-min pre-delivery features : provides the dynamic context of the courier right before they receive the batch.
    # order_id: The unique identifier for the delivery.
    # gps_points: The total count of raw GPS pings found in that 15-minute window.
    # speed_mean_15m: The average speed calculated from valid segments (filtered by your speed cap).
    # speed_std_15m: The standard deviation of the speed, representing how erratic the movement was.
    # distance_travelled_15m: The total sum of Euclidean distances between sequential GPS pings.
    # idle_fraction: The percentage of time the courier was recorded moving at a speed < 0.1 units (likely waiting or stationary).
    # coverage_ratio: A data quality metric representing the density of GPS pings relative to the expected sampling rate.
    return con.execute(f"WITH gps_window AS (SELECT d.order_id, g.gps_time, g.lat, g.lng, LAG(g.lat) OVER (PARTITION BY d.order_id ORDER BY g.gps_time) AS prev_lat, LAG(g.lng) OVER (PARTITION BY d.order_id ORDER BY g.gps_time) AS prev_lng, LAG(g.gps_time) OVER (PARTITION BY d.order_id ORDER BY g.gps_time) AS prev_time FROM delivery_tbl d JOIN parquet_scan('{gps_path}') g ON g.postman_id=d.delivery_user_id WHERE g.gps_time BETWEEN d.receipt_time-INTERVAL '{pre_min} minutes' AND d.receipt_time), motion AS (SELECT order_id, SQRT(POWER(lat-prev_lat,2)+POWER(lng-prev_lng,2)) AS dist, EXTRACT(EPOCH FROM (gps_time-prev_time)) AS dt FROM gps_window), speed_calc AS (SELECT order_id, dist, dt, CASE WHEN dt>1 AND dist<5000 AND (dist/dt)<={speed_cap} THEN dist/dt ELSE NULL END AS speed FROM motion) SELECT order_id, COUNT(*) AS gps_points, AVG(speed) AS speed_mean_15m, STDDEV(speed) AS speed_std_15m, SUM(dist) AS distance_travelled_15m, COUNT(CASE WHEN speed < 0.1 THEN 1 END) * 1.0 / COUNT(*) AS idle_fraction, COUNT(*) * 1.0 / ({pre_min} * 3) AS coverage_ratio FROM speed_calc GROUP BY order_id").pl()

def add_operational_and_distance_features(df, speed_cap):
    # EWMA and Remaining Haul Distance : captures courier 'momentum' and workload density. The EWMA (Exponentially Weighted Moving Average) serves as a proxy for the courier's individual efficiency 'profile', while the remaining_haul_distance quantifies the total spatial burden left in the current batch
    # Sort by user and time for historical profile
    df = df.sort(["delivery_user_id", "receipt_time", "order_id"])
    df = df.with_columns(
        pl.col("eta_mins").ewm_mean(half_life=10).over("delivery_user_id").shift(1).alias("courier_eta_ewm")
    )

    # Sort by batch_rank_dispatch
    df = df.sort(["batch_id", "batch_rank_dispatch"])

    # Calculate remaining haul distance (sum of distances from current rank to end of batch)

    # Static during ETA prediction.Must be recomputed online inside the simulator.
    df = df.with_columns([
        pl.col("pickup_destination_distance").reverse().cum_sum().over("batch_id").reverse().alias("remaining_haul_distance")
    ])
    return df

In [ ]:
# is_trajectory_available feature
def add_gps_missingness_flag(df):
    # Structural fill for missing GPS :

    # is_trajectory_available is binary , missing values filled with 0 --> imp for bandits

    df = df.with_columns(pl.col("speed_mean_15m").is_not_null().cast(pl.Int8).alias("is_trajectory_available"))

    return df.with_columns([pl.col("speed_mean_15m").fill_null(0), pl.col("distance_travelled_15m").fill_null(0), pl.col("gps_points").fill_null(0), pl.col("idle_fraction").fill_null(0), pl.col("coverage_ratio").fill_null(0)])

In [ ]:
# temporal features
def add_temporal_features(df, holidays, holiday_eve):
    # Cycles, Weekend, and Holiday flags
    df = df.with_columns([pl.col("receipt_time").dt.hour().alias("hour"), pl.col("receipt_time").dt.weekday().alias("weekday"), pl.col("receipt_time").dt.date().alias("receipt_date")])
    df = df.with_columns([(2*np.pi*pl.col("hour")/24).sin().alias("hour_sin"), (2*np.pi*pl.col("hour")/24).cos().alias("hour_cos")])
    df = df.with_columns([(pl.col("weekday")>=6).cast(pl.Int8).alias("is_weekend"), pl.col("receipt_date").cast(pl.Utf8).is_in(holidays).cast(pl.Int8).alias("is_holiday"), (pl.col("receipt_date").cast(pl.Utf8)==holiday_eve).cast(pl.Int8).alias("is_holiday_eve")])
    return df

In [ ]:
import polars as pl

# Analyze typecode distribution with counts and percentages
typecode_dist = (
    pl.from_pandas(wdata).group_by("typecode")
    .len()
    .with_columns(
        (pl.col("len") / pl.col("len").sum() * 100).alias("percentage")
    )
    .sort("len", descending=True)
)

print("Typecode Distribution (Top 15):")
display(typecode_dist.head(15))

print(f"\nTotal unique typecodes: {typecode_dist.height}")

In [ ]:
# Handling typecode -- dominant  2 typecodes , rest have minimal contribution hence grouped together
# typecode_cb = freq encoding for tree models ,
def add_typecode_encoding(df):
    # 1. Dynamically identify top 2 most frequent typecodes
    typecode_counts = df.group_by("typecode").len().sort("len", descending=True)

    # Handle case where there might be fewer than 2 unique codes
    top_codes = typecode_counts.get_column("typecode").head(2).to_list()

    type_1_code = top_codes[0] if len(top_codes) > 0 else "NONE_FOUND"
    type_2_code = top_codes[1] if len(top_codes) > 1 else "NONE_FOUND"

    # 2. Map to categories
    df = df.with_columns(
        pl.when(pl.col("typecode").is_null())
        .then(pl.lit("missing"))
        .when(pl.col("typecode") == type_1_code)
        .then(pl.lit("type_1"))
        .when(pl.col("typecode") == type_2_code)
        .then(pl.lit("type_2"))
        .otherwise(pl.lit("other"))
        .alias("typecode_grouped")
    )

    # 3. Add Frequency Encoding (typecode_cb) for tree-based models
    freq = df.group_by("typecode").len().rename({"len": "typecode_cb"})
    df = df.join(freq, on="typecode", how="left")

    # 4. One-Hot Encode (while keeping the original typecode column) ---> this is for causal discovery and linear models
    df = df.to_dummies(columns=["typecode_grouped"])

    return df

In [ ]:
# spatial congestion index creation
# converts coordinates into continuous coordinates , groups into 60 mins windows , sorted by day
# calculates density to count how many deliveries originates from same grid cell within same hour -- proxy for localised order density
# normalises afterwords - norm version is used for pcmci+

def add_spatial_congestion_v2(df, grid_size=500):
    # Grid-based SCI
    # grouping by truncate('1h') inherently includes the date, preventing cross-day leakage
    df = df.with_columns([
        (pl.col("receipt_lng") // grid_size).alias("grid_x"),
        (pl.col("receipt_lat") // grid_size).alias("grid_y"),
        pl.col("receipt_time").dt.truncate("1h").alias("time_window")
    ])

    # Calculate order density per grid-hour
    sci = df.group_by(["grid_x", "grid_y", "time_window"]).len().rename({"len": "spatial_congestion_daily"})

    df = df.join(sci, on=["grid_x", "grid_y", "time_window"], how="left")

    # Normalize the index to create a standardized scale for causal/linear models
    df = df.with_columns(
        ((pl.col("spatial_congestion_daily") - pl.col("spatial_congestion_daily").mean()) /
         pl.col("spatial_congestion_daily").std()).alias("spatial_congestion_norm")
    )
    return df

In [ ]:

# add the WSI data created in other csv


def add_weather(df, weather_csv, city_en):
    # Weather ASOF join
    # Ensures each delivery is matched with the latest weather report before receipt_time
    weather = pl.read_csv(weather_csv).with_columns(
        pl.col("datetime").str.strptime(pl.Datetime, strict=False).alias("weather_time")
    ).filter(pl.col("city") == city_en).sort("weather_time")

    return df.sort("receipt_time").join_asof(
        weather,
        left_on="receipt_time",
        right_on="weather_time",
        strategy="backward"
    ).drop("city")

## 5. Run All Cities

Executes the full pipeline for every city in `CITY_CONFIGS`.  
Results are stored in `city_results` (dict) and `city_batches` (dict).

In [ ]:
import os
import polars as pl
import duckdb
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE         = "/content/drive/MyDrive/ml/PROCESSED/matched/city_divided/"
WEATHER_BASE = "/content/drive/MyDrive/ml/weather-outputs/"
OUTPUT_DIR   = "/content/drive/MyDrive/ml/CORRECTEDv3/"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ── Pipeline hyper-parameters ─────────────────────────────────────────────────
PRE_MIN      = 15     # minutes of GPS history before receipt_time
MAX_GAP      = 30     # GPS staleness threshold (minutes)
GRID_SIZE    = 500    # spatial grid cell size (affine coordinate units ≈ 500 m)
WORKLOAD_CAP = 20     # workload cap for non-linear features

# ── Master City Loop ──────────────────────────────────────────────────────────
def build_city_features_v4(cfg: dict) -> pl.DataFrame:
    city_en = cfg["city_en"]
    delivery_file = BASE + cfg["delivery_file"]
    gps_file = BASE + cfg["gps_file"]
    weather_file = WEATHER_BASE + cfg["weather_file"]

    print(f"  [1/6] Loading delivery data: {cfg['delivery_file']}")
    df = load_delivery(delivery_file)
    df = add_euclidean_distance(df)
    df = add_batch_features(df)

    print(f"  [2/6] Computing GPS trajectory features (15m window)")
    con = duckdb.connect(database=':memory:', read_only=False)
    speed_cap = compute_speed_percentile(con, gps_file, percentile=0.99)
    con.execute("CREATE OR REPLACE TABLE delivery_tbl AS SELECT * FROM df")
    trajectory_features = compute_trajectory_features(con, gps_file, speed_cap, PRE_MIN)
    df = df.join(trajectory_features, on="order_id", how="left")

    print(f"  [3/6] Joining last-known courier positions")
    gps_df_polars = pl.read_parquet(gps_file)
    df = courier_snapshot(df, gps_df_polars)
    df = filter_stale_gps(df, MAX_GAP)

    print(f"  [4/6] Adding operational and temporal features")
    df = add_operational_and_distance_features(df, speed_cap)
    df = add_gps_missingness_flag(df)
    df = add_temporal_features(df, cfg["holidays"], cfg["holiday_eve"])

    print(f"  [5/6] Encoding typecodes and spatial congestion")
    df = add_typecode_encoding(df)
    df = add_spatial_congestion_v2(df, GRID_SIZE)

    print(f"  [6/6] Joining weather data")
    df = add_weather(df, weather_file, city_en)

    con.close()
    return df

city_results_v4 = {}
trajectory_cols = ['speed_mean_15m', 'speed_std_15m', 'distance_travelled_15m', 'gps_points']

for cfg in CITY_CONFIGS:
    print(f"\n>>> Processing City: {cfg['city_en']}")
    df_city = build_city_features_v4(cfg)

    # Final null cleaning for model readiness
    df_city = df_city.with_columns([
        pl.col('courier_eta_ewm').fill_null(0),
        pl.col('typecode_cb').fill_null(0),
        *[pl.col(c).fill_null(0) for c in trajectory_cols]
    ])

    city_results_v4[cfg['city_en']] = df_city
    out_path = os.path.join(OUTPUT_DIR, f"delivery_features_{cfg['city_en'].lower()}.parquet")
    df_city.write_parquet(out_path)
    print(f"  SUCCESS: Saved {df_city.height:,} rows to {out_path}")

# Combine all cities into one master feature set
combined_v4 = pl.concat(list(city_results_v4.values()), how="diagonal")
combined_v4.write_parquet(os.path.join(OUTPUT_DIR, "all_cities_delivery_features_v4.parquet"))
print(f"\n✅ Stage 1 Complete. Total rows processed: {combined_v4.height:,}")

In [ ]:
print(f"Columns for Shanghai (v4):\n{city_results_v4['Shanghai'].columns}")

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

def run_workload_diagnostic(df, city_name):
    print(f'\n--- Diagnostic: {city_name} ---')

    # 1. Correlation with target (commented out as feature is removed)
    # correlation = df.select(pl.corr('active_queue_at_receipt', 'eta_mins')).item()
    # print(f'Pearson Correlation (Workload vs ETA): {correlation:.4f}')

    # 2. Visualizing the impact of workload on ETA (commented out as feature is removed)
    # agg_df = (
    #     df.with_columns(pl.col('active_queue_at_receipt').clip(upper_bound=15))
    #     .group_by('active_queue_at_receipt')
    #     .agg(pl.mean('eta_mins').alias('avg_eta'))
    #     .sort('active_queue_at_receipt')
    # )

    # plt.figure(figsize=(8, 4))
    # sns.lineplot(data=agg_df.to_pandas(), x='active_queue_at_receipt', y='avg_eta', marker='o')
    # plt.title(f'Impact of Workload on Delivery Time ({city_name})')
    # plt.xlabel('Concurrent Orders (Workload)')
    # plt.ylabel('Average ETA (mins)')
    # plt.grid(True, alpha=0.3)
    # plt.show()

# Run for available cities in memory if processed (commented out as feature is removed)
# for city, df in city_results_v4.items():
#     run_workload_diagnostic(df, city)

In [ ]:
import os

# The 31 columns requested by the user
FINAL_FEATURES = [
    "order_id", "batch_id", "delivery_user_id", "receipt_time",
    "courier_eta_ewm",
    "batch_size", "batch_rank_dispatch",
    "pickup_destination_distance", "remaining_haul_distance",
    "gps_points", "speed_mean_15m", "speed_std_15m", "distance_travelled_15m", "is_trajectory_available",
    "typecode", "typecode_cb",
    "spatial_congestion_daily", "spatial_congestion_norm",
    "hour_sin", "hour_cos", "is_weekend", "is_holiday", "is_holiday_eve",
    "temperature_2m", "precipitation", "windspeed_10m", "WSI",
    "eta_mins"
]

# Process and save each city individually
for city_name, df_city in city_results_v4.items():
    slug = city_name.lower()

    # Filter to available requested columns
    available_cols = [c for c in FINAL_FEATURES if c in df_city.columns]
    df_filtered = df_city.select(available_cols)

    # Save per user's naming convention
    city_output_path = os.path.join(OUTPUT_DIR, f"delivery_lvl_{slug}_v4.parquet")
    df_filtered.write_parquet(city_output_path)

    print(f"Saved {city_name} (filtered) to: {city_output_path} | Shape: {df_filtered.shape}")

# Also update the combined filtered dataset for convenience
available_comb = [c for c in FINAL_FEATURES if c in combined_v4.columns]
df_final_v4_comb = combined_v4.select(available_comb)
comb_output_path = os.path.join(OUTPUT_DIR, "all_cities_v4_filtered.parquet")
df_final_v4_comb.write_parquet(comb_output_path)

print(f"\nCombined filtered dataset saved to: {comb_output_path}")
display(df_final_v4_comb.head())

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# import polars as pl
# import numpy as np

# def plot_eta_by_city(city_results, clip_pct=0.95):
#     fig, ax = plt.subplots(figsize=(10, 5))
#     colors = {"Shanghai": "#e74c3c", "Hangzhou": "#2980b9", "Chongqing": "#27ae60"}
#     for city, df in city_results.items():
#         cap = df["eta_mins"].quantile(clip_pct)
#         data = df.filter(pl.col("eta_mins") <= cap)["eta_mins"].to_numpy()
#         med = float(np.median(data))
#         sns.kdeplot(data, ax=ax, label=f"{city} (median={med:.0f} min)", color=colors.get(city), fill=True, alpha=0.25)
#     ax.set_title(f"ETA Distribution by City (≤P{int(clip_pct*100)})")
#     ax.legend()
#     plt.tight_layout()
#     plt.show()

# def plot_saturation_by_city(city_results):
#     fig, axes = plt.subplots(1, len(city_results), figsize=(15, 4))
#     colors = {"Shanghai": "#e74c3c", "Hangzhou": "#2980b9", "Chongqing": "#27ae60"}
#     for ax, (city, df) in zip(axes, city_results.items()):
#         plot_df = df.with_columns(pl.col("active_queue_at_receipt").clip(upper_bound=20)).group_by("active_queue_at_receipt").agg(pl.mean("eta_mins").alias("mean_eta")).sort("active_queue_at_receipt").to_pandas()
#         ax.bar(plot_df["active_queue_at_receipt"], plot_df["mean_eta"], color=colors.get(city, "steelblue"), alpha=0.8)
#         ax.axvline(x=14.5, color="red", linestyle="--", alpha=0.6)
#         ax.set_title(city)
#     plt.suptitle("Workload Saturation Curve — All Cities", y=1.02)
#     plt.tight_layout()
#     plt.show()

# def plot_feature_corr_heatmap(city_results):
#     key_features = ["active_queue_at_receipt", "batch_rank_dispatch", "speed_mean_15m", "pickup_destination_distance", "spatial_congestion_norm", "WSI", "eta_mins"]
#     fig, axes = plt.subplots(1, len(city_results), figsize=(18, 6))
#     for ax, (city, df) in zip(axes, city_results.items()):
#         avail = [c for c in key_features if c in df.columns]
#         corr_mat = df.select(avail).to_pandas().corr()
#         sns.heatmap(corr_mat, ax=ax, cmap="coolwarm", center=0, annot=True, fmt=".2f", vmin=-1, vmax=1)
#         ax.set_title(city)
#     plt.tight_layout()
#     plt.show()

# print("Generating updated visualizations for Stage 1 results...")
# plot_eta_by_city(city_results_v4)
# plot_saturation_by_city(city_results_v4)
# plot_feature_corr_heatmap(city_results_v4)

In [ ]:
# def plot_eta_by_city(city_results: dict, clip_pct: float = 0.95) -> None:
#     """ETA KDE overlay for all cities on one axis."""
#     fig, ax = plt.subplots(figsize=(10, 5))
#     colors = {"Shanghai": "#e74c3c", "Hangzhou": "#2980b9", "Chongqing": "#27ae60"}
#     for city, df in city_results.items():
#         cap  = df["eta_mins"].quantile(clip_pct)
#         data = df.filter(pl.col("eta_mins") <= cap)["eta_mins"].to_numpy()
#         med  = float(np.median(data))
#         sns.kdeplot(data, ax=ax, label=f"{city} (median={med:.0f} min)",
#                     color=colors.get(city), fill=True, alpha=0.25)
#     ax.set_xlabel("ETA (mins)")
#     ax.set_ylabel("Density")
#     ax.set_title(f"ETA Distribution by City (≤P{int(clip_pct*100)})")
#     ax.legend()
#     plt.tight_layout()
#     plt.show()


# def plot_saturation_by_city(city_results: dict) -> None:
#     """Workload saturation curves — 1×3 subplot grid. Uses active_queue_at_receipt (v2)."""
#     fig, axes = plt.subplots(1, len(city_results), figsize=(15, 4), sharey=False)
#     colors = {"Shanghai": "#e74c3c", "Hangzhou": "#2980b9", "Chongqing": "#27ae60"}
#     for ax, (city, df) in zip(axes, city_results.items()):
#         plot_df = (
#             df.with_columns(pl.col("active_queue_at_receipt").clip(upper_bound=20))
#             .group_by("active_queue_at_receipt")
#             .agg(pl.mean("eta_mins").alias("mean_eta"))
#             .sort("active_queue_at_receipt")
#             .to_pandas()
#         )
#         ax.bar(plot_df["active_queue_at_receipt"], plot_df["mean_eta"],
#                color=colors.get(city, "steelblue"), alpha=0.8)
#         ax.axvline(x=14.5, color="red", linestyle="--", alpha=0.6)
#         ax.set_title(city)
#         ax.set_xlabel("active_queue_at_receipt (clipped at 20)")
#         ax.set_ylabel("Mean ETA (mins)")
#     plt.suptitle("Workload Saturation Curve — All Cities", fontsize=13, y=1.02)
#     plt.tight_layout()
#     plt.show()


# def plot_feature_corr_heatmap(city_results: dict) -> None:
#     """Side-by-side correlation heatmaps (key features vs eta_mins)."""
#     key_features = [
#         "active_queue_at_receipt", "workload_capped",
#         "batch_rank_dispatch", "batch_rank_capped",
#         "speed_mean", "pickup_destination_distance",
#         "spatial_congestion_norm", "gps_gap_min",
#         "WSI", "hour_sin", "is_weekend", "is_holiday",
#         "delivery_sequence_daily", "eta_mins",
#     ]
#     fig, axes = plt.subplots(1, len(city_results), figsize=(18, 8))
#     for ax, (city, df) in zip(axes, city_results.items()):
#         avail = [c for c in key_features if c in df.columns]
#         corr_mat = df.select(avail).corr().to_pandas()
#         corr_mat.columns = avail
#         corr_mat.index   = avail
#         sns.heatmap(corr_mat, ax=ax, cmap="coolwarm", center=0,
#                     annot=True, fmt=".2f", annot_kws={"size": 7},
#                     linewidths=0.4, vmin=-1, vmax=1)
#         ax.set_title(city)
#         ax.tick_params(axis="x", rotation=45)
#         ax.tick_params(axis="y", rotation=0)
#     plt.suptitle("Pearson Correlation Heatmap — Key Features", fontsize=13, y=1.01)
#     plt.tight_layout()
#     plt.show()


# # ── Run cross-city plots ──────────────────────────────────────────────────────
# plot_eta_by_city(city_results)
# plot_saturation_by_city(city_results)
# plot_feature_corr_heatmap(city_results)

### 5.1 Post-Processing & Additional Sanity Checks
Handling missing trajectory data by filling with 0 and verifying feature consistency.

## 8. Advanced EDA — Spatial, Interaction & SHAP

Five specialized diagnostics for the thesis documentation:
1. **Courier Ridgeline** — ETA distribution across top-N couriers (joypy)
2. **Temporal Heatmap** — Day × Hour mean ETA (Qingming Festival visible)
3. **Spatial Overlay** — grid-based delivery density vs mean ETA bottlenecks
4. **Workload–Batch Interaction** — 2D heatmap of workload × batch rank
5. **SHAP Feature Importance** — XGBoost + beeswarm plot per city

In [ ]:
!pip install shap joypy xgboost --quiet
import shap
import joypy
import xgboost
print(f"SHAP {shap.__version__} | XGBoost {xgboost.__version__}")

In [ ]:
import shap
import joypy
import xgboost
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import numpy as np

def plot_courier_ridgeline(features, city_en, top_n=20):
    top_couriers = features.group_by("delivery_user_id").len().sort("len", descending=True).head(top_n).get_column("delivery_user_id").to_list()
    plot_df = features.filter(pl.col("delivery_user_id").is_in(top_couriers)).to_pandas()
    joypy.joyplot(
        plot_df, by="delivery_user_id", column="eta_mins",
        range_style="own", grid="y", linewidth=1, legend=False,
        title=f"Courier ETA Distribution (Top {top_n}) — {city_en}",
        colormap=plt.cm.viridis, alpha=0.6, figsize=(10, 8)
    )
    plt.show()

def plot_temporal_heatmap(features, city_en):
    heatmap_data = features.with_columns([pl.col("receipt_time").dt.day().alias("day"), pl.col("receipt_time").dt.hour().alias("hour")]).group_by(["day", "hour"]).agg(pl.mean("eta_mins").alias("mean_eta")).to_pandas()
    pivot_df = heatmap_data.pivot(index="hour", columns="day", values="mean_eta").sort_index()
    plt.figure(figsize=(14, 7))
    sns.heatmap(pivot_df, cmap="YlOrRd", annot=True, fmt=".0f", cbar_kws={"label": "Mean ETA (mins)"})
    plt.title(f"Temporal ETA Heatmap: Day vs Hour — {city_en}")
    plt.tight_layout()
    plt.show()

def plot_spatial_overlay(features, city_en):
    spatial_stats = features.group_by(["grid_x", "grid_y"]).agg([pl.len().alias("density"), pl.mean("eta_mins").alias("mean_eta")]).to_pandas()
    fig, ax = plt.subplots(1, 2, figsize=(18, 7))
    ax[0].scatter(spatial_stats["grid_x"], spatial_stats["grid_y"],
                        c=spatial_stats["density"], cmap="viridis", s=20)
    ax[0].set_title(f"Density — {city_en}")
    ax[1].scatter(spatial_stats["grid_x"], spatial_stats["grid_y"],
                        c=spatial_stats["mean_eta"], cmap="Reds", s=20)
    ax[1].set_title(f"ETA Bottlenecks — {city_en}")
    plt.show()

def plot_workload_batch_interaction(features, city_en):
    # This function depends on 'active_queue_at_receipt' which is being removed.
    # Commenting out its implementation.
    print(f"Plotting workload-batch interaction for {city_en} skipped: 'active_queue_at_receipt' feature removed.")
    # interaction_df = features.group_by(["active_queue_at_receipt", "batch_rank_dispatch"]).agg(pl.mean("eta_mins").alias("mean_eta")).to_pandas()
    # pivot_df = interaction_df.pivot(index="active_queue_at_receipt", columns="batch_rank_dispatch", values="mean_eta").sort_index()
    # plt.figure(figsize=(12, 8))
    # sns.heatmap(pivot_df.iloc[:20, :15], annot=True, fmt=".1f", cmap="rocket_r")
    # plt.title(f"Workload x Batch Rank Interaction — {city_en}")
    # plt.show()

def plot_shap_relevance(features, city_en):
    # This function depends on 'active_queue_at_receipt' which is being removed.
    # Modifying model_features to exclude it.
    model_features = ["batch_size", "batch_rank_dispatch", "speed_mean_15m", "pickup_destination_distance", "WSI", "spatial_congestion_norm"]
    avail_cols = [c for c in model_features if c in features.columns]
    X = features.select(avail_cols).to_pandas().fillna(0)
    y = features["eta_mins"].to_numpy()
    model = xgboost.XGBRegressor(n_estimators=50, max_depth=5, random_state=42).fit(X, y)
    explainer = shap.Explainer(model, X)
    shap_values = explainer(X)
    plt.figure(figsize=(10, 6))
    shap.plots.beeswarm(shap_values, show=False)
    plt.title(f"SHAP Importance — {city_en}")
    plt.show()

def run_advanced_eda(city_results):
    for city_name, df in city_results.items():
        print(f"\nProcessing {city_name}...")
        plot_courier_ridgeline(df, city_name)
        plot_temporal_heatmap(df, city_name)
        plot_spatial_overlay(df, city_name)
        plot_workload_batch_interaction(df, city_name) # This call will now print a skipped message
        plot_shap_relevance(df, city_name)

run_advanced_eda(city_results_v4)

#Intermediate Features that are dropped

- receipt_time_window
- _same_aoi_count
- batch_cum_centroid_lng
- batch_cum_centroid_lat
- hour
- receipt_date
- grid_x, grid_y
- time_window

## 9. Feature Summary Table

Compact per-city summary for thesis documentation.

In [ ]:
def feature_summary(city_results_dict: dict) -> None:
    """
    Print null counts and ETA statistics for all numeric features per city.
    Useful for the thesis dataset chapter and appendix.
    """
    for city, df in city_results_dict.items():
        print(f"\n{'─'*60}")
        print(f"  {city}  |  {df.height:,} rows × {len(df.columns)} cols")
        print(f"{'─'*60}")

        # Identify numeric columns using Polars types
        num_cols = [
            c for c, t in df.schema.items()
            if t.is_numeric()
        ]

        # Pre-calculate null counts for performance
        null_counts = df.select([pl.col(c).null_count().alias(c) for c in num_cols])

        print(f"  eta_mins  mean={df['eta_mins'].mean():.1f}  "
              f"median={df['eta_mins'].median():.1f}  "
              f"p95={df['eta_mins'].quantile(0.95):.1f}")

        print(f"\n  {'Feature':<40} {'Nulls':<10}")
        print(f"  {'-'*50}")
        for col in sorted(num_cols):
            n_null = null_counts[col].item()
            tag = f"⚠ {n_null}" if n_null > 0 else "0"
            print(f"  {col:<40} {tag:<10}")

# Run the summary using the latest city results
feature_summary(city_results_v4)

In [ ]:
# Verify the feature logic for a sample batch
verification_cols = [
    "batch_id",
    "batch_size",
    "batch_rank_actual",
    "pickup_destination_distance",
    "eta_mins"
]

# Filter for a multi-order batch to see the sequence logic using the latest combined data
try:
    sample_batch_id = combined_v4.filter(pl.col("batch_size") > 2).get_column("batch_id")[0]

    verification_df = (
        combined_v4.filter(pl.col("batch_id") == sample_batch_id)
        .select([c for c in verification_cols if c in combined_v4.columns])
        .sort("batch_rank_actual")
    )

    print(f"Verification for Batch ID: {sample_batch_id}")
    display(verification_df)

    # Summary stats for key features across the whole dataset
    print("\nGlobal Summary of Key Features:")
    display(combined_v4.select(["batch_size", "pickup_destination_distance"]).describe())
except Exception as e:
    print(f"Verification failed: {e}. Ensure combined_v4 is defined and contains batch data.")

---
***
---


---

BATCH AGGREGATION

---




Batch aggregation is a crucial step in the feature pipeline, transforming order-level data into batch-level insights. This process aims to capture characteristics relevant to a courier's entire delivery batch rather than individual orders. Each feature is aggregated according to its nature, ensuring the resulting batch-level features accurately reflect the operational context.

Key aspects of batch aggregation include:

-   **Identification & Context**: Features that are constant within a batch, such as `delivery_user_id`, `receipt_time` (of the first order in the batch), and environmental factors like `WSI`, are typically taken as the `first()` value within the batch.
-   **Size and Diversity**: Metrics like `batch_size` (total orders in the batch) and `batch_grid_cells_unique` (number of distinct spatial grid cells covered by the batch) are computed to understand the scope and complexity of the batch.
-   **Operational Means**: Features that vary across orders within a batch, such as `speed_mean_15m` and `pickup_destination_distance`, are averaged (`mean()`) to represent the batch's overall operational characteristics.
-   **Spatial Congestion Aggregates**: For features like `spatial_congestion_daily`, both `mean()` and `std()` are calculated to capture both the average congestion and its variability across the batch's delivery locations.
-   **Targets**: The target variable, `eta_mins`, is also aggregated to the batch level, typically by taking the `mean()`, `max()`, and `std()` to provide a comprehensive view of batch delivery times.

This aggregation ensures that Stage 2 (Causal Discovery / RL) receives a dataset where each row represents a delivery batch, with features specifically engineered to capture batch-level dynamics and influences.

 Step 2B — Aggregated Delivery Features
    (correct aggregation per feature type)

        Feature                  Aggregation     Reason
        ───────────────────────  ──────────────  ─────────────────────────────────
        workload_causal          first()         Constant within batch (same receipt_time)
        courier_local_load       first()         Constant within batch (same courier+hour)
        WSI                      first()         Constant within batch (same time+city)
        hour_sin, hour_cos       first()         Constant within batch
        is_weekend, is_holiday   first()         Constant within batch
        speed_mean_15m           mean()          Varies by order; true operational mean
        speed_std_15m            mean()          Varies by order
        idle_fraction            mean()          Varies by order
        coverage_ratio           mean()          Varies by order
        spatial_congestion_daily mean() + std()  Varies by POI grid cell → mean + cv
        rolling_congestion_3h    first()         Hour-level; constant within batch
        pickup_destination_dist  mean()          Varies by order
        remaining_haul_distance  mean()          Varies by order
        delivery_sequence_daily  mean()          Varies by batch_rank within batch
        courier_eta_ewm          first()         Courier-level; constant within batch
        typecode_cb              mean()          Varies by order


In [ ]:
# TODO :


# batch_size
# is_singleton_batch

# aoi entropy : Let pi = batch size / orders in AOI i ​

# Then H=− i∑pilog(pi)

# The spatial heterogeneity of the batch.

# Low entropy → deliveries concentrated into one/few AOIs.
# High entropy → deliveries spread across many AOIs.


In [ ]:
def add_nearest_neighbor_distance(df: pl.DataFrame) -> pl.DataFrame:
    """
    Calculates the distance to the geographically closest other delivery within the same batch.
    Singleton batches are assigned a distance of 0.
    """
    # Prepare coordinates for distance calculation
    # We use a self-join approach within batches for vectorized distance computation

    # 1. Create a simplified dataframe for pairing
    coords = df.select(["batch_id", "order_id", "poi_lat", "poi_lng"])

    # 2. Join orders with all other orders in the same batch
    pairs = coords.join(coords, on="batch_id", suffix="_other")

    # 3. Filter out self-comparisons
    pairs = pairs.filter(pl.col("order_id") != pl.col("order_id_other"))

    # 4. Compute Euclidean distances between pairs
    pairs = pairs.with_columns(
        ((pl.col("poi_lat") - pl.col("poi_lat_other")).pow(2) +
         (pl.col("poi_lng") - pl.col("poi_lng_other")).pow(2)).sqrt().alias("dist")
    )

    # 5. Find the minimum distance for each order
    nn_dist = pairs.group_by(["batch_id", "order_id"]).agg(
        pl.col("dist").min().alias("nearest_neighbor_distance")
    )

    # 6. Join back to main dataframe and fill nulls (singletons) with 0
    return df.join(nn_dist, on=["batch_id", "order_id"], how="left").with_columns(
        pl.col("nearest_neighbor_distance").fill_null(0.0)
    )

print("Updating datasets with nearest_neighbor_distance...")

# Update individual city results
for city in city_results_v4:
    city_results_v4[city] = add_nearest_neighbor_distance(city_results_v4[city])
    print(f"  ✓ {city} updated")

# Update combined master set
combined_v4 = add_nearest_neighbor_distance(combined_v4)

# Verification of a multi-order batch
display(combined_v4.filter(pl.col("batch_size") > 1)
        .select(["batch_id", "order_id", "poi_lat", "poi_lng", "nearest_neighbor_distance"])
        .head(5))

In [ ]:
import polars as pl
import numpy as np

def add_batch_features(df: pl.DataFrame) -> pl.DataFrame:
    # 1. Basic Batch Stats
    batch_stats = df.group_by("batch_id").agg([
        pl.len().alias("batch_size"),
        pl.col("last_delivery_duration_clipped").mean().alias("batch_last_delivery_duration_mean")
    ]).with_columns([
        (pl.col("batch_size") == 1).cast(pl.Int8).alias("is_singleton_batch")
    ])

    # 2. AOI Entropy Calculation : Shannon Entropy: A measure of how much new information you gain when a particular event occurs.
    # H = - sum(pi * log(pi)) where pi is the share of orders in AOI i within the batch
    aoi_counts = (
        df.group_by(["batch_id", "aoi_id"])
        .len()
        .join(batch_stats.select(["batch_id", "batch_size"]), on="batch_id")
        .with_columns((
            pl.col("len") / pl.col("batch_size")
        ).alias("pi"))
    )

    aoi_entropy = (
        aoi_counts.with_columns((
            pl.col("pi") * pl.col("pi").log(base=2)
        ).alias("p_log_p"))
        .group_by("batch_id")
        .agg((-pl.col("p_log_p").sum()).alias("batch_aoi_entropy"))
    )

    # 3. Merge features back
    advanced_batch_features = batch_stats.join(aoi_entropy, on="batch_id")

    return advanced_batch_features

# Generate the features from our combined dataset
batch_features_v5 = add_advanced_batch_features(combined_v4)

print("Advanced Batch Features Created.")
print(f"Sample rows from batch_features_v5:")
display(batch_features_v5.head())

# Update the main batch dataframe if needed
batch_df_v4 = batch_df_v4.join(batch_features_v5.drop(["batch_size"]), on="batch_id", how="left")

### 2.17 `aggregate_to_batch_level` — Stage 2A Entry Point

This function aggregates the order-level features into a batch-level dataset.

**Key Metrics:**
- **Spatial Diversity:** `batch_grid_cells_unique` counts distinct grid cells.
- **Operational state:** `workload_causal` and `WSI` (constant per batch).
- **Performance metrics:** Aggregated speed, distance, and ETA (mean/max/std).

In [ ]:
def aggregate_to_batch_level(df: pl.DataFrame) -> pl.DataFrame:
    """
    Aggregates order-level features to batch-level (unique batch_id).
    Follows the specific aggregation rules for Stage 2A.
    """
    # 1. Define aggregation expressions
    batch_aggs = [
        # Identification & Context (Constant per batch)
        pl.col("delivery_user_id").first(),
        pl.col("receipt_time").first(),
        pl.col("city").first().alias("city") if "city" in df.columns else pl.lit(None).alias("city"),

        # Features constant within batch
        pl.col("courier_eta_ewm").first(),
        pl.col("WSI").first(),
        pl.col("hour_sin").first(),
        pl.col("hour_cos").first(),
        pl.col("is_weekend").first(),
        pl.col("is_holiday").first(),
        pl.col("is_holiday_eve").first(),

        # Size and Diversity
        pl.len().alias("batch_size"),
        pl.col("grid_x").n_unique().alias("batch_grid_cells_unique"),

        # Operational Means (Varies by order)
        pl.col("speed_mean_15m").mean().alias("speed_mean_15m_mean"),
        pl.col("speed_std_15m").mean().alias("speed_std_15m_mean"),
        pl.col("distance_travelled_15m").mean().alias("distance_travelled_15m_mean"),
        pl.col("pickup_destination_distance").mean().alias("pickup_destination_distance_mean"),
        pl.col("remaining_haul_distance").mean().alias("remaining_haul_distance_mean"),
        pl.col("typecode_cb").mean().alias("typecode_cb_mean"),

        # Spatial Congestion Aggregates
        pl.col("spatial_congestion_daily").mean().alias("spatial_congestion_daily_mean"),
        pl.col("spatial_congestion_daily").std().alias("spatial_congestion_daily_std"),
        pl.col("spatial_congestion_norm").mean().alias("spatial_congestion_norm_mean"),
        pl.col("spatial_congestion_norm").std().alias("spatial_congestion_norm_std"),

        # Targets
        pl.col("eta_mins").mean().alias("eta_mean"),
        pl.col("eta_mins").max().alias("eta_max"),
        pl.col("eta_mins").std().alias("eta_std")
    ]

    batch_df = (
        df.group_by("batch_id")
        .agg(batch_aggs)
        .with_columns([
            # Add a flag for singleton batches to handle edge cases
            (pl.col("batch_size") == 1).cast(pl.Int8).alias("is_singleton_batch"),
            # Fill std dev nulls (from single-order batches) with 0
            pl.col("eta_std").fill_null(0),
            pl.col("spatial_congestion_daily_std").fill_null(0),
            pl.col("spatial_congestion_norm_std").fill_null(0)
        ])
        .sort(["delivery_user_id", "receipt_time"])
    )

    return batch_df

# Run aggregation for the combined dataset
batch_df_v4 = aggregate_to_batch_level(combined_v4)

# Save the aggregated batch data
batch_output_path = os.path.join(OUTPUT_DIR, "batch_aggregated_v4.parquet")
batch_df_v4.write_parquet(batch_output_path)

print(f"✅ Batch Aggregation Complete.")
print(f"Aggregated Shape: {batch_df_v4.shape[0]:,} batches x {batch_df_v4.shape[1]} columns")
display(batch_df_v4.head())

In [ ]:
### Batch-Level Feature Definitions

The following features are computed after aggregating order-level data to the unique `batch_id` level. These capture the operational complexity and spatial distribution of a courier's simultaneous assignments.

| Feature Group | Column Name | Description |
| :--- | :--- | :--- |
| **Structural** | `batch_size` | Total number of orders assigned to the courier in a single dispatch window. |
| | `is_singleton_batch` | Binary flag (1 if `batch_size == 1`, else 0). Identifies non-batched deliveries. |
| **Spatial Complexity** | `batch_aoi_entropy` | Shannon Entropy of Area of Interest (AOI) distribution. High values indicate geographic dispersion; 0 indicates all orders are at one location. |
| | `batch_grid_cells_unique` | Count of unique 500m x 500m spatial grid cells visited within the batch. |
| **Operational Sequence** | `batch_last_delivery_duration_mean` | Average time (mins) between consecutive deliveries within the batch (clipped at 99th percentile). |
| **Movement Context** | `speed_mean_15m_mean` | Average speed of the courier across all orders in the batch during the 15 minutes prior to receipt. |
| | `distance_travelled_15m_mean` | Average distance covered by the courier across the batch window prior to start. |
| **Congestion** | `spatial_congestion_daily_mean` | Average number of other deliveries originating from the same grid-hour cells as this batch. |
| **Environmental** | `WSI` | Weather Severity Index (composite of precipitation, wind, and temp) mapped to the batch start time. |
| **Targets** | `eta_mean` | The primary dependent variable: average delivery time (receipt to sign) for all orders in the batch. |

### Batch-Level Feature Definitions

The following features are computed after aggregating order-level data to the unique `batch_id` level. These capture the operational complexity and spatial distribution of a courier's simultaneous assignments.

| Feature Group | Column Name | Description |
| :--- | :--- | :--- |
| **Structural** | `batch_size` | Total number of orders assigned to the courier in a single dispatch window. |
| | `is_singleton_batch` | Binary flag (1 if `batch_size == 1`, else 0). Identifies non-batched deliveries. |
| **Spatial Complexity** | `batch_aoi_entropy` | Shannon Entropy of Area of Interest (AOI) distribution. High values indicate geographic dispersion; 0 indicates all orders are at one location. |
| | `batch_grid_cells_unique` | Count of unique 500m x 500m spatial grid cells visited within the batch. |
| **Operational Sequence** | `batch_last_delivery_duration_mean` | Average time (mins) between consecutive deliveries within the batch (clipped at 99th percentile). |
| **Movement Context** | `speed_mean_15m_mean` | Average speed of the courier across all orders in the batch during the 15 minutes prior to receipt. |
| | `distance_travelled_15m_mean` | Average distance covered by the courier across the batch window prior to start. |
| **Congestion** | `spatial_congestion_daily_mean` | Average number of other deliveries originating from the same grid-hour cells as this batch. |
| **Environmental** | `WSI` | Weather Severity Index (composite of precipitation, wind, and temp) mapped to the batch start time. |
| **Targets** | `eta_mean` | The primary dependent variable: average delivery time (receipt to sign) for all orders in the batch. |

In [ ]:
import os

# Loop through each city and save individual batch-level SCM parquets
for city_name, df_city in city_results_v4.items():
    slug = city_name.lower()

    # Aggregate city-specific data to batch level
    batch_city_df = aggregate_to_batch_level(df_city)

    # Define output path using the requested naming convention
    output_path = os.path.join(OUTPUT_DIR, f"batch_scm_{slug}.parquet")

    # Save to parquet
    batch_city_df.write_parquet(output_path)

    print(f"✅ Generated {city_name} Batch SCM: {output_path} | {batch_city_df.height:,} batches")

display(batch_city_df.head(3))

In [ ]:
## Causal Resources

### Verification Checks for Stage 2 Readiness
We verify that the city-specific SCM parquets are correctly formatted and contain no missing values in key causal features.

In [ ]:
import polars as pl
import os

# Verification loop for the three SCM files
scm_files = ['batch_scm_shanghai.parquet', 'batch_scm_hangzhou.parquet', 'batch_scm_chongqing.parquet']

for file_name in scm_files:
    path = os.path.join(OUTPUT_DIR, file_name)
    df_check = pl.read_parquet(path)

    print(f"\n--- Verification: {file_name} ---")
    print(f"Rows: {df_check.height:,} | Columns: {len(df_check.columns)}")

    # Check for nulls in crucial columns
    # Removed 'active_queue_at_receipt' as it is no longer part of the features
    nulls = df_check.select([
        pl.col("eta_mean").null_count().alias("null_eta"),
        pl.col("WSI").null_count().alias("null_wsi"),
        pl.col("batch_size").null_count().alias("null_size")
    ])
    print(f"Null Summary: {nulls.to_dicts()[0]}")

    # Descriptive stats for the target
    print(f"ETA Mean Range: [{df_check['eta_mean'].min():.1f}, {df_check['eta_mean'].max():.1f}] mins")

In [ ]:
import polars as pl
import os

causal_features = [
    "batch_size",
    "batch_aoi_entropy",
    "batch_grid_cells_unique",
    "speed_mean_15m_mean",
    "WSI",
    "eta_mean"
]

scm_files = ['batch_scm_shanghai.parquet', 'batch_scm_hangzhou.parquet', 'batch_scm_chongqing.parquet']

for file_name in scm_files:
    path = os.path.join(OUTPUT_DIR, file_name)
    if not os.path.exists(path):
        continue
    df = pl.read_parquet(path)

    print(f"\n{'-'*60}")
    print(f"Feature Statistics: {file_name}")
    print(f"{' ' * 5} (Total Batches: {df.height:,})")
    print(f"{'-'*60}")

    summary_rows = []
    for feat in causal_features:
        if feat in df.columns:
            # Cast stats to float64 to ensure schema consistency during concat
            stats = df.select([
                pl.lit(feat).alias("feature"),
                pl.col(feat).min().cast(pl.Float64).alias("min"),
                pl.col(feat).max().cast(pl.Float64).alias("max"),
                pl.col(feat).mean().cast(pl.Float64).alias("mean"),
                pl.col(feat).null_count().cast(pl.Float64).alias("nulls")
            ])
            summary_rows.append(stats)

    if summary_rows:
        final_summary = pl.concat(summary_rows)
        display(final_summary)